In [ ]:
import os
import shutil
from tqdm import tqdm

# ✅ Configuration
config = {
    0: "Arachnida",
    1: "Formicidae",
    2: "Coleoptera",
    3: "Nematocera",
    4: "Brachycera",
    5: "Apoidea",
    6: "Syraphidae"
}

# ✅ Paths
base_path = "/content/drive/MyDrive/data_5"  # Change this to your dataset path
train_dir = os.path.join(base_path, "training")
val_dir = os.path.join(base_path, "val")
merged_dir = os.path.join(base_path, "merged_dataset")
os.makedirs(merged_dir, exist_ok=True)

# ✅ Create class-specific folders inside the merged directory
for cls_name in config.values():
    class_path = os.path.join(merged_dir, cls_name)
    os.makedirs(os.path.join(class_path, "images"), exist_ok=True)
    os.makedirs(os.path.join(class_path, "labels"), exist_ok=True)

def merge_and_separate(src_dir, merged_dir):
    image_count = 0
    label_count = 0

    images_dir = os.path.join(src_dir, "images")
    labels_dir = os.path.join(src_dir, "labels")

    for img_name in tqdm(os.listdir(images_dir), desc=f"Processing {src_dir}"):
        if img_name.endswith(".jpg"):
            img_path = os.path.join(images_dir, img_name)
            label_name = img_name.replace(".jpg", ".txt")
            label_path = os.path.join(labels_dir, label_name)

            # Read the label file to determine the class
            if os.path.exists(label_path):
                with open(label_path, "r") as f:
                    lines = f.readlines()

                if len(lines) > 0:
                    # Get the class index from the first line
                    cls_index = int(lines[0].split()[0])
                    cls_name = config.get(cls_index, "Unknown")

                    # If class is valid, copy to respective class folder
                    if cls_name != "Unknown":
                        dest_img_dir = os.path.join(merged_dir, cls_name, "images")
                        dest_label_dir = os.path.join(merged_dir, cls_name, "labels")
                        shutil.copy(img_path, os.path.join(dest_img_dir, img_name))
                        shutil.copy(label_path, os.path.join(dest_label_dir, label_name))
                        image_count += 1
                        label_count += 1
                    else:
                        print(f"❗ Unknown class index in file: {label_path}")
                else:
                    print(f"⚠️ Empty label file: {label_path}")
            else:
                print(f"❗ Label not found for image: {img_name}")

    return image_count, label_count

# ✅ Merge training and validation data
train_images, train_labels = merge_and_separate(train_dir, merged_dir)
val_images, val_labels = merge_and_separate(val_dir, merged_dir)

# ✅ Summary
print("\n📊 Dataset Merge and Separation Summary:")
print(f"Training Images Merged: {train_images}")
print(f"Training Labels Merged: {train_labels}")
print(f"Validation Images Merged: {val_images}")
print(f"Validation Labels Merged: {val_labels}")
print(f"Total Images Merged: {train_images + val_images}")
print(f"Total Labels Merged: {train_labels + val_labels}")
print(f"Merged dataset saved to: {merged_dir}")


Processing /content/drive/MyDrive/data_5/training: 100%|██████████| 2973/2973 [29:51<00:00,  1.66it/s]
Processing /content/drive/MyDrive/data_5/val: 100%|██████████| 330/330 [02:58<00:00,  1.85it/s]


📊 Dataset Merge and Separation Summary:
Training Images Merged: 2973
Training Labels Merged: 2973
Validation Images Merged: 330
Validation Labels Merged: 330
Total Images Merged: 3303
Total Labels Merged: 3303
Merged dataset saved to: /content/drive/MyDrive/data_5/merged_dataset


In [ ]:
import os
import shutil
from tqdm import tqdm

# ✅ Configuration
config = {
    0: "Apoidea",
    1: "Arachnida",
    2: "Brachycera",
    3: "Coleoptera",
    4: "Formicidae",
    5: "Nematocera",
    6: "Syraphidae"
}

# ✅ Paths
base_path = "/content/drive/MyDrive/data_5/merged_dataset"  # Update this to your dataset path
output_path = "/content/drive/MyDrive/data_5/merged_dataset/verified_merged_dataset"
os.makedirs(output_path, exist_ok=True)
os.makedirs(os.path.join(output_path, "images"), exist_ok=True)
os.makedirs(os.path.join(output_path, "labels"), exist_ok=True)

# ✅ Function to verify images and labels and reindex them
def verify_and_reindex(class_name, class_index):
    class_dir = os.path.join(base_path, class_name)
    img_dir = os.path.join(class_dir, "images")
    lbl_dir = os.path.join(class_dir, "labels")
    verified_count = 0

    if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
        print(f"🚫 Missing images or labels directory for class: {class_name}")
        return 0

    for img_name in tqdm(os.listdir(img_dir), desc=f"🔍 Verifying {class_name}"):
        if not img_name.endswith(".jpg"):
            continue

        img_path = os.path.join(img_dir, img_name)
        label_name = img_name.replace(".jpg", ".txt")
        label_path = os.path.join(lbl_dir, label_name)

        # Check if label file exists
        if not os.path.exists(label_path):
            print(f"⚠️ Missing label for image: {img_name}")
            continue

        # Read and reindex label file
        with open(label_path, "r") as f:
            lines = f.readlines()

        # Check if label file is not empty
        if len(lines) == 0:
            print(f"⚠️ Empty label file: {label_path}")
            continue

        # Overwrite label with correct class index
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                print(f"⚠️ Corrupted label file: {label_path}")
                continue

            # Replace the class index with the correct one
            parts[0] = str(class_index)
            new_lines.append(" ".join(parts))

        # Save reindexed label to the merged output directory
        output_img_path = os.path.join(output_path, "images", img_name)
        output_lbl_path = os.path.join(output_path, "labels", label_name)

        # Copy image to merged folder
        shutil.copy(img_path, output_img_path)

        # Save reindexed label
        with open(output_lbl_path, "w") as f:
            f.write("\n".join(new_lines))

        verified_count += 1

    print(f"✅ Verified and reindexed {verified_count} images for class {class_name}.")
    return verified_count

# ✅ Process each class
total_verified = 0
for idx, cls_name in config.items():
    verified = verify_and_reindex(cls_name, idx)
    total_verified += verified

print("\n📊 Verification and Reindexing Summary:")
print(f"Total verified images: {total_verified}")
print(f"Dataset saved to: {output_path}")


🔍 Verifying Apoidea: 100%|██████████| 114/114 [00:59<00:00,  1.90it/s]


✅ Verified and reindexed 114 images for class Apoidea.


🔍 Verifying Arachnida: 100%|██████████| 1205/1205 [09:21<00:00,  2.15it/s]


✅ Verified and reindexed 1205 images for class Arachnida.


🔍 Verifying Brachycera: 100%|██████████| 241/241 [01:53<00:00,  2.12it/s]


...

✅ Verified and reindexed 576 images for class Formicidae.


🔍 Verifying Nematocera: 100%|██████████| 401/401 [03:01<00:00,  2.21it/s]


✅ Verified and reindexed 401 images for class Nematocera.


🔍 Verifying Syraphidae: 100%|██████████| 1/1 [00:01<00:00,  1.00s/it]

✅ Verified and reindexed 1 images for class Syraphidae.

📊 Verification and Reindexing Summary:
Total verified images: 3051
Dataset saved to: /content/drive/MyDrive/data_5/merged_dataset/verified_merged_dataset


In [3]:
import os
import shutil
from tqdm import tqdm
from collections import defaultdict

# ✅ Paths
current_dataset_path = "/content/drive/MyDrive/data_5/merged_dataset/verified_merged_dataset"
old_train_images_path = "/content/drive/MyDrive/hand-picked-data-final-split-corrected_again/train/images"
old_train_labels_path = "/content/drive/MyDrive/hand-picked-data-final-split-corrected_again/train/labels"
old_val_images_path = "/content/drive/MyDrive/hand-picked-data-final-split-corrected_again/val/images"
old_val_labels_path = "/content/drive/MyDrive/hand-picked-data-final-split-corrected_again/val/labels"

# ✅ Output Path
output_path = "/content/drive/MyDrive/data_5/final_merged_dataset"
os.makedirs(os.path.join(output_path, "images"), exist_ok=True)
os.makedirs(os.path.join(output_path, "labels"), exist_ok=True)

# ✅ Class names as per configuration
class_names = {
    0: "Apoidea",
    1: "Arachnida",
    2: "Brachycera",
    3: "Coleoptera",
    4: "Formicidae",
    5: "Nematocera",
    6: "Syraphidae"
}

# ✅ Counters
image_count = defaultdict(int)
background_count = 0

# ✅ Function to merge images and labels
def merge_images_and_labels(source_images, source_labels):
    global background_count
    for img_name in tqdm(os.listdir(source_images), desc=f"🔄 Merging images from {source_images}"):
        if not img_name.endswith(".jpg"):
            continue

        img_path = os.path.join(source_images, img_name)
        label_name = img_name.replace(".jpg", ".txt")
        label_path = os.path.join(source_labels, label_name)

        # Check if label file exists
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()

            if len(lines) == 0:
                # Empty label file, treat as background
                background_count += 1
            else:
                # Check the class in the label file
                classes_in_image = set(int(line.split()[0]) for line in lines)
                for cls in classes_in_image:
                    image_count[cls] += 1
        else:
            # No label file, treat as background
            background_count += 1

        # Copy image and label
        shutil.copy(img_path, os.path.join(output_path, "images", img_name))
        if os.path.exists(label_path):
            shutil.copy(label_path, os.path.join(output_path, "labels", label_name))

    print(f"✅ Merged images from {source_images} successfully.")

# ✅ Merging Current Dataset
print("🔁 Merging current dataset...")
merge_images_and_labels(os.path.join(current_dataset_path, "images"), os.path.join(current_dataset_path, "labels"))

# ✅ Merging Old Train Dataset
print("🔁 Merging old train dataset...")
merge_images_and_labels(old_train_images_path, old_train_labels_path)

# ✅ Merging Old Val Dataset
print("🔁 Merging old val dataset...")
merge_images_and_labels(old_val_images_path, old_val_labels_path)

# ✅ Summary
print("\n📊 Merging Summary:")
total_images = sum(image_count.values()) + background_count
print(f"Total Merged Images: {total_images}")
for cls, count in sorted(image_count.items()):
    print(f"Class {cls} ({class_names.get(cls, 'Unknown')}): {count} images")
print(f"Background-only Images (no label files): {background_count}")

print(f"\n✅ Merged dataset saved to: {output_path}")


🔁 Merging current dataset...


🔄 Merging images from /content/drive/MyDrive/data_5/merged_dataset/verified_merged_dataset/images: 100%|██████████| 3051/3051 [28:27<00:00,  1.79it/s]


✅ Merged images from /content/drive/MyDrive/data_5/merged_dataset/verified_merged_dataset/images successfully.
🔁 Merging old train dataset...


🔄 Merging images from /content/drive/MyDrive/hand-picked-data-final-split-corrected_again/train/images: 100%|██████████| 1592/1592 [12:06<00:00,  2.19it/s]


✅ Merged images from /content/drive/MyDrive/hand-picked-data-final-split-corrected_again/train/images successfully.
🔁 Merging old val dataset...


🔄 Merging images from /content/drive/MyDrive/hand-picked-data-final-split-corrected_again/val/images: 100%|██████████| 402/402 [03:01<00:00,  2.22it/s]

✅ Merged images from /content/drive/MyDrive/hand-picked-data-final-split-corrected_again/val/images successfully.

📊 Merging Summary:
Total Merged Images: 5045
Class 0 (Apoidea): 166 images
Class 1 (Arachnida): 1369 images
Class 2 (Brachycera): 616 images
Class 3 (Coleoptera): 762 images
Class 4 (Formicidae): 899 images
Class 5 (Nematocera): 835 images
Class 6 (Syraphidae): 182 images
Background-only Images (no label files): 216

✅ Merged dataset saved to: /content/drive/MyDrive/data_5/final_merged_dataset


In [1]:
import os
import shutil
from tqdm import tqdm
from collections import defaultdict

# ✅ Paths
merged_dataset_path = "/content/drive/MyDrive/data_5/final_merged_dataset"  # Your existing merged dataset path
background_only_images_path = "/content/drive/MyDrive/data_5/merged_dataset/Background_images_sensibe"  # Path to background-only images

# ✅ Output Path
output_path = "/content/drive/MyDrive/data_5/final_merged_dataset"
os.makedirs(os.path.join(output_path, "images"), exist_ok=True)

# ✅ Counter
background_count = 0

# ✅ Merging Background-Only Images
print("🔁 Merging background-only images...")
for img_name in tqdm(os.listdir(background_only_images_path), desc="Processing Background Images"):
    if img_name.endswith(".jpg"):
        src_path = os.path.join(background_only_images_path, img_name)
        dst_path = os.path.join(output_path, "images", img_name)
        shutil.copy(src_path, dst_path)
        background_count += 1

# ✅ Summary
print("\n📊 Final Merging Summary:")
print(f"Total Background-only Images Added: {background_count}")
print(f"\n✅ Merged dataset saved to: {output_path}")


🔁 Merging background-only images...


Processing Background Images: 100%|██████████| 191/191 [00:06<00:00, 28.05it/s]


📊 Final Merging Summary:
Total Background-only Images Added: 183

✅ Merged dataset saved to: /content/drive/MyDrive/data_5/final_merged_dataset


In [2]:
import os
from collections import defaultdict

# ✅ Paths
merged_dataset_path = "/content/drive/MyDrive/data_5/final_merged_dataset"

# ✅ Class names as per your configuration
class_names = {
    0: "Apoidea",
    1: "Arachnida",
    2: "Brachycera",
    3: "Coleoptera",
    4: "Formicidae",
    5: "Nematocera",
    6: "Syraphidae"
}

# ✅ Counters
class_counts = defaultdict(int)
background_count = 0

# ✅ Counting Images and Background Only Images
print("🔍 Counting images per class and background-only images...")

image_files = [f for f in os.listdir(os.path.join(merged_dataset_path, "images")) if f.endswith(".jpg")]

for img_name in image_files:
    label_name = img_name.replace(".jpg", ".txt")
    label_path = os.path.join(merged_dataset_path, "labels", label_name)

    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            labels = f.readlines()
            for line in labels:
                class_id = int(line.split()[0])
                class_counts[class_id] += 1
    else:
        # Image without label file, count as background-only
        background_count += 1

# ✅ Display Summary
print("\n📊 Final Image Count Summary:")
for class_id, count in class_counts.items():
    print(f"  {class_names[class_id]}: {count} images")
print(f"  Background-only images: {background_count}")


🔍 Counting images per class and background-only images...

📊 Final Image Count Summary:
  Apoidea: 166 images
  Arachnida: 1469 images
  Brachycera: 623 images
  Coleoptera: 794 images
  Formicidae: 1017 images
  Nematocera: 876 images
  Syraphidae: 182 images
  Background-only images: 399


In [3]:
import os
from collections import defaultdict

# ✅ Paths
merged_dataset_path = "/content/drive/MyDrive/data_5/final_merged_dataset"

# ✅ Class names as per your configuration
class_names = {
    0: "Apoidea",
    1: "Arachnida",
    2: "Brachycera",
    3: "Coleoptera",
    4: "Formicidae",
    5: "Nematocera",
    6: "Syraphidae"
}

# ✅ Counters
class_counts = defaultdict(int)
background_count = 0
mismatched_files = []

# ✅ Counting Images and Background Only Images
print("🔍 Verifying images, labels, and background-only images...")

image_files = [f for f in os.listdir(os.path.join(merged_dataset_path, "images")) if f.endswith(".jpg")]

for img_name in image_files:
    label_name = img_name.replace(".jpg", ".txt")
    label_path = os.path.join(merged_dataset_path, "labels", label_name)

    if os.path.exists(label_path):
        try:
            with open(label_path, 'r') as f:
                labels = f.readlines()
                if len(labels) == 0:
                    # Empty label file - count as background-only
                    background_count += 1
                else:
                    for line in labels:
                        class_id = int(line.split()[0])
                        if class_id in class_names:
                            class_counts[class_id] += 1
                        else:
                            print(f"❗ Unrecognized class ID {class_id} in file: {label_name}")
        except Exception as e:
            print(f"⚠️ Error reading label file {label_name}: {e}")
            mismatched_files.append(label_name)
    else:
        # Image without label file, count as background-only
        background_count += 1

# ✅ Identify mismatched image-label pairs (files without corresponding labels)
label_files = [f for f in os.listdir(os.path.join(merged_dataset_path, "labels")) if f.endswith(".txt")]
unmatched_labels = [lf for lf in label_files if lf.replace(".txt", ".jpg") not in image_files]

# ✅ Display Summary
print("\n📊 Final Image Count Summary:")
for class_id, count in class_counts.items():
    print(f"  {class_names[class_id]}: {count} images")
print(f"  Background-only images: {background_count}")

# ✅ Display Mismatches
if mismatched_files or unmatched_labels:
    print("\n⚠️ Mismatched or Unpaired Files:")
    if mismatched_files:
        print(f"  ➡️ Label files with errors: {len(mismatched_files)}")
        for file in mismatched_files:
            print(f"    - {file}")
    if unmatched_labels:
        print(f"  ➡️ Unmatched label files: {len(unmatched_labels)}")
        for file in unmatched_labels:
            print(f"    - {file}")
else:
    print("\n✅ All image-label pairs are correctly matched!")


🔍 Verifying images, labels, and background-only images...

📊 Final Image Count Summary:
  Apoidea: 166 images
  Arachnida: 1469 images
  Brachycera: 623 images
  Coleoptera: 794 images
  Formicidae: 1017 images
  Nematocera: 876 images
  Syraphidae: 182 images
  Background-only images: 399

✅ All image-label pairs are correctly matched!


In [4]:
import os
import shutil
from tqdm import tqdm
from collections import defaultdict

# ✅ Paths
current_dataset_path = "/content/drive/MyDrive/data_5/merged_dataset/verified_merged_dataset"
old_train_images_path = "/content/drive/MyDrive/hand-picked-data-final-split-corrected_again/train/images"
old_train_labels_path = "/content/drive/MyDrive/hand-picked-data-final-split-corrected_again/train/labels"
old_val_images_path = "/content/drive/MyDrive/hand-picked-data-final-split-corrected_again/val/images"
old_val_labels_path = "/content/drive/MyDrive/hand-picked-data-final-split-corrected_again/val/labels"

# ✅ Output Path
output_path = "/content/drive/MyDrive/data_5/final_merged_dataset_again"
os.makedirs(os.path.join(output_path, "images"), exist_ok=True)
os.makedirs(os.path.join(output_path, "labels"), exist_ok=True)

# ✅ Class names as per configuration
class_names = {
    0: "Apoidea",
    1: "Arachnida",
    2: "Brachycera",
    3: "Coleoptera",
    4: "Formicidae",
    5: "Nematocera",
    6: "Syraphidae"
}

# ✅ Counters
image_count = defaultdict(int)
background_count = 0

# ✅ Function to merge images and labels
def merge_images_and_labels(source_images, source_labels):
    global background_count
    for img_name in tqdm(os.listdir(source_images), desc=f"🔄 Merging images from {source_images}"):
        if not img_name.endswith(".jpg"):
            continue

        img_path = os.path.join(source_images, img_name)
        label_name = img_name.replace(".jpg", ".txt")
        label_path = os.path.join(source_labels, label_name)

        # Check if label file exists
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()

            if len(lines) == 0:
                # Empty label file, treat as background
                background_count += 1
            else:
                # Check the class in the label file
                classes_in_image = set(int(line.split()[0]) for line in lines)
                for cls in classes_in_image:
                    image_count[cls] += 1
        else:
            # No label file, treat as background
            background_count += 1

        # Copy image and label
        shutil.copy(img_path, os.path.join(output_path, "images", img_name))
        if os.path.exists(label_path):
            shutil.copy(label_path, os.path.join(output_path, "labels", label_name))

    print(f"✅ Merged images from {source_images} successfully.")

# ✅ Merging Current Dataset
print("🔁 Merging current dataset...")
merge_images_and_labels(os.path.join(current_dataset_path, "images"), os.path.join(current_dataset_path, "labels"))

# ✅ Merging Old Train Dataset
print("🔁 Merging old train dataset...")
merge_images_and_labels(old_train_images_path, old_train_labels_path)

# ✅ Merging Old Val Dataset
print("🔁 Merging old val dataset...")
merge_images_and_labels(old_val_images_path, old_val_labels_path)

# ✅ Summary
print("\n📊 Merging Summary:")
total_images = sum(image_count.values()) + background_count
print(f"Total Merged Images: {total_images}")
for cls, count in sorted(image_count.items()):
    print(f"Class {cls} ({class_names.get(cls, 'Unknown')}): {count} images")
print(f"Background-only Images (no label files): {background_count}")

print(f"\n✅ Merged dataset saved to: {output_path}")


🔁 Merging current dataset...


🔄 Merging images from /content/drive/MyDrive/data_5/merged_dataset/verified_merged_dataset/images: 100%|██████████| 3051/3051 [18:34<00:00,  2.74it/s]


✅ Merged images from /content/drive/MyDrive/data_5/merged_dataset/verified_merged_dataset/images successfully.
🔁 Merging old train dataset...


🔄 Merging images from /content/drive/MyDrive/hand-picked-data-final-split-corrected_again/train/images: 100%|██████████| 1592/1592 [08:04<00:00,  3.29it/s]


✅ Merged images from /content/drive/MyDrive/hand-picked-data-final-split-corrected_again/train/images successfully.
🔁 Merging old val dataset...


🔄 Merging images from /content/drive/MyDrive/hand-picked-data-final-split-corrected_again/val/images: 100%|██████████| 402/402 [01:53<00:00,  3.55it/s]

✅ Merged images from /content/drive/MyDrive/hand-picked-data-final-split-corrected_again/val/images successfully.

📊 Merging Summary:
Total Merged Images: 5045
Class 0 (Apoidea): 166 images
Class 1 (Arachnida): 1369 images
Class 2 (Brachycera): 616 images
Class 3 (Coleoptera): 762 images
Class 4 (Formicidae): 899 images
Class 5 (Nematocera): 835 images
Class 6 (Syraphidae): 182 images
Background-only Images (no label files): 216

✅ Merged dataset saved to: /content/drive/MyDrive/data_5/final_merged_dataset_again


In [5]:
import os
import shutil
from tqdm import tqdm
from collections import defaultdict

# ✅ Paths
merged_dataset_path = "/content/drive/MyDrive/data_5/final_merged_dataset_again"
background_images_path = "/content/drive/MyDrive/data_5/merged_dataset/Background_images_sensibe"

# ✅ Counters
background_count = 0

# ✅ Function to merge background-only images
def merge_background_images(source_images):
    global background_count
    added_count = 0
    for img_name in tqdm(os.listdir(source_images), desc=f"🔄 Merging background images from {source_images}"):
        if not img_name.endswith(".jpg"):
            continue

        img_path = os.path.join(source_images, img_name)
        # Check if the image already exists in the merged dataset to avoid duplicates
        if not os.path.exists(os.path.join(merged_dataset_path, "images", img_name)):
            shutil.copy(img_path, os.path.join(merged_dataset_path, "images", img_name))
            background_count += 1
            added_count += 1

    print(f"✅ Merged background images from {source_images} successfully.")
    return added_count

# ✅ Merging Background Images
print("🔁 Merging additional background-only images...")
added_background_count = merge_background_images(background_images_path)

# ✅ Summary
print("\n📊 Merging Summary:")
print(f"Newly Added Background-only Images: {added_background_count}")
print(f"Total Background-only Images after merging: {background_count}")

# ✅ Count Images per Class and Background
class_names = {
    0: "Apoidea",
    1: "Arachnida",
    2: "Brachycera",
    3: "Coleoptera",
    4: "Formicidae",
    5: "Nematocera",
    6: "Syraphidae"
}
image_count = defaultdict(int)
background_count = 0

# ✅ Counting Images per Class and Background
for img_name in tqdm(os.listdir(os.path.join(merged_dataset_path, "images")), desc="🔍 Counting images"):
    if not img_name.endswith(".jpg"):
        continue
    label_name = img_name.replace(".jpg", ".txt")
    label_path = os.path.join(merged_dataset_path, "labels", label_name)
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            lines = f.readlines()
        if len(lines) == 0:
            background_count += 1
        else:
            classes_in_image = set(int(line.split()[0]) for line in lines)
            for cls in classes_in_image:
                image_count[cls] += 1
    else:
        background_count += 1

# ✅ Final Count Summary
print("\n📊 Final Count Summary:")
total_images = sum(image_count.values()) + background_count
print(f"Total Merged Images: {total_images}")
for cls, count in sorted(image_count.items()):
    print(f"Class {cls} ({class_names.get(cls, 'Unknown')}): {count} images")
print(f"Background-only Images (no label files): {background_count}")

print(f"\n✅ Merged dataset updated at: {merged_dataset_path}")


🔁 Merging additional background-only images...


🔄 Merging background images from /content/drive/MyDrive/data_5/merged_dataset/Background_images_sensibe: 100%|██████████| 191/191 [00:03<00:00, 48.17it/s]


✅ Merged background images from /content/drive/MyDrive/data_5/merged_dataset/Background_images_sensibe successfully.

📊 Merging Summary:
Newly Added Background-only Images: 183
Total Background-only Images after merging: 183


🔍 Counting images: 100%|██████████| 5228/5228 [00:15<00:00, 342.75it/s]


📊 Final Count Summary:
Total Merged Images: 5228
Class 0 (Apoidea): 166 images
Class 1 (Arachnida): 1369 images
Class 2 (Brachycera): 616 images
Class 3 (Coleoptera): 762 images
Class 4 (Formicidae): 899 images
Class 5 (Nematocera): 835 images
Class 6 (Syraphidae): 182 images
Background-only Images (no label files): 399

✅ Merged dataset updated at: /content/drive/MyDrive/data_5/final_merged_dataset_again
